# Inference Tokenization

Today’s drill explores how transformer models balance computational throughput and memory efficiency during text generation.

We will build a simulation harness to model:

- KV-cache memory mechanics
- INT4 weight quantization error
- static vs dynamic batching inefficiency

### Core Architecture (Constraints)
- No GPU / PyTorch
- Pure Python + NumPy simulation only
- Deterministic arithmetic modeling (no stochastic behavior)
- Match inference-engine style math (vLLM / llama.cpp style)

## Engineering Requirements

### KV-Cache Memory Engine Simulation

When a transformer generates text token-by-token, it stores Key and Value tensors for every token at every layer.

This avoids recomputing attention states but increases memory usage linearly with sequence length.

### KV Cache Memory Formula

$$
KV\ Cache\ Size\ (Bytes) =
\frac{2 \times b \times s \times l \times h_{kv} \times d \times p_{bits}}{8}
$$

Where:

- $b$ = batch size  
- $s$ = sequence length  
- $l$ = number of layers  
- $h_{kv}$ = KV heads  
- $d$ = head dimension  
- $p_{bits}$ = precision in bits (FP16 = 16, FP32 = 32)

### Exected Output
```
KV CACHE (GB): <float>  
MSE: <float>  
STATIC COST: <int>  
DYNAMIC COST: <int>  
WASTED TOKENS: <int>
```

### Imports

In [37]:
import numpy as np
import math

### KV Cache Implementation

In [38]:
def kv_cache_bytes(b, s, l, h_kv, d, p):
    """
    Calculate the number of bytes required for the KV cache.

    Parameters:
    b (int): Batch size
    s (int): Sequence length
    l (int): Number of layers
    h_kv (int): Number of attention heads for keys and values
    d (int): Dimension of each head
    p (int): Precision in bits (e.g., 16 for float16, 32 for float32)

    Returns:
    int: Total number of bytes required for the KV cache
    """
    # Calculate the number of elements in the KV cache
    num_elements = b * s * l * h_kv * d * 2  # Multiply by 2 for keys and values

    # Calculate the total number of bits and convert to bytes
    total_bits = num_elements * p
    total_bytes = total_bits / 8  # Convert bits to bytes

    return total_bytes

def kv_cache_gb(b, s, l, h_kv, d, p_bits):
    """
    Calculate the number of gigabytes required for the KV cache.

    Parameters:
    b (int): Batch size
    s (int): Sequence length
    l (int): Number of layers
    h_kv (int): Number of attention heads for keys and values
    d (int): Dimension of each head
    p_bits (int): Precision in bits (e.g., 16 for float16, 32 for float32)

    Returns:
    float: Total number of gigabytes required for the KV cache
    """
    # Calculate the total bytes using the kv_cache_bytes function
    total_bytes = kv_cache_bytes(b, s, l, h_kv, d, p_bits)

    # Convert bytes to gigabytes
    total_gb = total_bytes / (1024 ** 3)  # Convert bytes to GB

    return total_gb

#### KV Cache Verification

In [39]:
# sanity check: KV cache should scale linearly with sequence length
small = kv_cache_bytes(1, 128, 32, 8, 128, 16)
large = kv_cache_bytes(1, 256, 32, 8, 128, 16)

print("Small KV:", small)
print("Large KV:", large)

assert large == 2 * small, "KV cache scaling failed (should be linear in sequence length)"

Small KV: 16777216.0
Large KV: 33554432.0


#### KV Cache Unit Verification

In [40]:
gb = kv_cache_gb(1, 4096, 32, 8, 128, 16)

print("KV GB:", gb)

assert gb > 0
assert gb < 1000, "KV cache value looks unreasonably large"

KV GB: 0.5


### Weight Quantization Arithmetic Simulator

We simulate INT4 symmetric quantization using block-wise scaling of FP16 weights.

### Input Weight Vector

In [41]:
weights = np.array([-1.42, 0.35, 2.88, -0.91, 0.12, -2.15, 1.74, 0.05], dtype=np.float32)

### Quantization Steps

Step 1: Max absolute value

$$
\alpha = \max(|w|)
$$

Step 2: Scale factor

$$
S = \frac{\alpha}{7}
$$

Step 3: Quantization

$$
q = \text{clip}\left(\text{round}\left(\frac{w}{S}\right), -7, 7\right)
$$

Step 4: Dequantization

$$
\hat{w} = q \cdot S
$$

Step 5: MSE

$$
MSE = \frac{1}{n} \sum (w - \hat{w})^2
$$

### Quantization

In [42]:
alpha = np.max(np.abs(weights))
S = alpha / 7

q = np.round(weights / S)
q = np.clip(q, -7, 7)

w_hat = q * S

mse = np.mean((weights - w_hat) ** 2)

#### Quantization Verification

In [43]:
# reconstruction sanity check
reconstruction_error = np.abs(weights - w_hat)

print("Max error:", np.max(reconstruction_error))
print("MSE:", mse)

# sanity: quantized values must be within INT4 range
assert np.all(q >= -7) and np.all(q <= 7), "INT4 clipping failed"

# sanity: reconstruction should not explode
assert mse < 1.0, "Quantization error unexpectedly large"

Max error: 0.18571413
MSE: 0.010033662


### Dynamic Batching Throughput Analyzer

Compare:

Static Batching
- Pads all sequences → wasted compute

Dynamic Batching
- Processes only real tokens → no padding waste

### Sample Seed Benchmark Dataset

In [44]:
llama_8b_config = {
    "num_layers": 32,
    "num_kv_heads": 8,
    "head_dim": 128,
    "bytes_per_param_fp16": 2, 
    "bytes_per_param_int4": 0.5
}

mock_batch_requests = [
    {"input_tokens": 128, "max_gen_tokens": 32},
    {"input_tokens": 512, "max_gen_tokens": 64},
    {"input_tokens": 32,  "max_gen_tokens": 128} 
]

### Static Batching

Let:

$$
L_{max} = \max(input\_tokens + max\_gen\_tokens)
$$

Waste computation:

$$
Waste = \sum (L_{max} - L_i)
$$

In [45]:
def static_batch(batch):
    lengths = [r["input_tokens"] + r["max_gen_tokens"] for r in batch]
    max_len = max(lengths)

    total_compute = 0
    wasted = 0

    for l in lengths:
        total_compute += max_len
        wasted += (max_len - l)

    return total_compute, wasted

### Dynamic Batching

$$
Cost = \sum (input\_tokens + max\_gen\_tokens)
$$

In [46]:
def dynamic_batch(batch):
    return sum(r["input_tokens"] + r["max_gen_tokens"] for r in batch)

#### Batch Verification

In [47]:
lengths = [r["input_tokens"] + r["max_gen_tokens"] for r in mock_batch_requests]

static_cost, wasted = static_batch(mock_batch_requests)
dynamic_cost = dynamic_batch(mock_batch_requests)

print("Lengths:", lengths)
print("Static:", static_cost)
print("Dynamic:", dynamic_cost)
print("Waste:", wasted)

# sanity checks
assert static_cost >= dynamic_cost, "Static batching should never be cheaper"
assert wasted >= 0, "Waste cannot be negative"

Lengths: [160, 576, 160]
Static: 1728
Dynamic: 896
Waste: 832


### Throughput Estimation

Inference engines are often evaluated using token throughput.

A simple throughput metric is:

$$
TPS = \frac{Total\ Tokens}{Execution\ Time}
$$

where:

- $TPS$ = tokens generated per second
- Total Tokens = workload processed
- Execution Time = simulated runtime in seconds

For this simulation, we will use a fixed execution time to estimate throughput.

In [48]:
def tokens_per_second(total_tokens, execution_time_seconds):
    return total_tokens / execution_time_seconds

#### Throughput Verification

In [49]:
simulated_runtime = 0.25

tps = tokens_per_second(
    dynamic_cost,
    simulated_runtime
)

print("TOKENS/SEC:", tps)

assert tps > 0, "Throughput must be positive"

TOKENS/SEC: 3584.0


### Execution Harness

Runs:

- KV cache VRAM simulation
- INT4 quantization error
- batching efficiency comparison

In [50]:
def run_simulation():
    kv = kv_cache_gb(1, 4096, 32, 8, 128, 16)  # FP16 = 16 bits

    alpha = np.max(np.abs(weights))
    S = alpha / 7
    q = np.round(weights / S)
    q = np.clip(q, -7, 7)
    w_hat = q * S
    mse = np.mean((weights - w_hat) ** 2)

    static_cost, wasted = static_batch(mock_batch_requests)
    dynamic_cost = dynamic_batch(mock_batch_requests)

    simulated_runtime = 0.25

    tps = tokens_per_second(
        dynamic_cost,
        simulated_runtime
    )

    print("KV CACHE (GB):", kv)
    print("MSE:", mse)
    print("TOKENS/SEC:", tps)
    print("STATIC COST:", static_cost)
    print("DYNAMIC COST:", dynamic_cost)
    print("WASTED TOKENS:", wasted)

### Execute Run Simulation

In [51]:
run_simulation()

KV CACHE (GB): 0.5
MSE: 0.010033662
TOKENS/SEC: 3584.0
STATIC COST: 1728
DYNAMIC COST: 896
WASTED TOKENS: 832


- KV cache memory uses bit-level precision modeling
- INT4 quantization introduces bounded reconstruction error
- Static batching wastes compute due to padding
- Dynamic batching improves throughput efficiency
- Token throughput can be estimated using processed tokens divided by execution time